In [ ]:
!pip install crewai crewai-tools
pip install anthropic
pip install gradio

In [ ]:
import pandas as pd
df = pd.read_csv('hard_leetcode.csv')
df.head()

In [ ]:
tasks = df.description

In [ ]:
task_list = tasks.values
task_list = task_list[:50]


In [ ]:
import random
import json
from typing import List, Dict
from anthropic import Anthropic, HUMAN_PROMPT, AI_PROMPT

def create_anthropic_client(api_key: str) -> Anthropic:
    return Anthropic(api_key=api_key)

class AnthropicAgent:
    def __init__(self, name: str, client: Anthropic):
        self.name = name
        self.client = client

    def process(self, input_data: str) -> str:
        print(f"{self.name} processing: {input_data}")
        response = self.client.completions.create(
            model="claude-2",
            max_tokens_to_sample=1000,
            prompt=f"{HUMAN_PROMPT} {input_data}{AI_PROMPT}",
        )
        return response.completion.strip()

class InitialPromptGenerator(AnthropicAgent):
    def process(self, problem_description: str) -> str:
        prompt = f"""Given the following LeetCode hard problem description, create an initial prompt that would guide an AI to write Python code to solve this problem:

Problem description:
{problem_description}

Generate the initial prompt:"""
        return super().process(prompt)

class CodeGenerator(AnthropicAgent):
    def process(self, task: str) -> str:
        prompt = f"Write a Python function to solve the following LeetCode hard problem: {task}. Provide only the code, no explanations."
        return super().process(prompt)

class CodeAnalyzer(AnthropicAgent):
    def process(self, code: str) -> Dict[str, str]:
        prompt = f"""Analyze the following Python code that solves a LeetCode hard problem and provide:
        1. A score from 0 to 10 based on space-time complexity and function logic, where 10 is the best.
        2. Brief feedback on the code.
        3. The time complexity of the solution (e.g., O(n), O(n^2), etc.).

        Code:
        {code}

        Output ONLY the result in JSON format with 'score', 'feedback', and 'time_complexity' fields. Do not include any other text."""
        response = super().process(prompt)

        try:
            return json.loads(response)
        except json.JSONDecodeError:
            print(f"Error parsing JSON from: {response}")
            # If JSON parsing fails, try to extract information manually
            import re
            score_match = re.search(r'score"?\s*:\s*(\d+)', response)
            feedback_match = re.search(r'feedback"?\s*:\s*"?([^"}\n]+)', response)
            complexity_match = re.search(r'time_complexity"?\s*:\s*"?([^"}\n]+)', response)

            return {
                "score": int(score_match.group(1)) if score_match else 5,
                "feedback": feedback_match.group(1).strip() if feedback_match else "Error analyzing code",
                "time_complexity": complexity_match.group(1).strip() if complexity_match else "Unknown"
            }

class PromptOptimizer(AnthropicAgent):
    def process(self, original_prompt: str, code: str, feedback: str) -> str:
        prompt = f"""Original prompt: {original_prompt}
        Generated code: {code}
        Feedback: {feedback}

        Based on this information, provide an improved version of the original prompt that would lead to better code for solving this LeetCode hard problem. Output only the improved prompt, no explanations."""
        return super().process(prompt)

def agentic_workflow(problem_description: str, anthropic_client: Anthropic, max_iterations: int = 3, improvement_threshold: int = 2) -> Dict[str, str]:
    agent0 = InitialPromptGenerator("Agent 0 (Initial Prompt Generator)", anthropic_client)
    agent1 = PromptOptimizer("Agent 1 (Prompt Optimizer)", anthropic_client)
    agent2 = CodeGenerator("Agent 2 (Code Generator)", anthropic_client)
    agent3 = CodeAnalyzer("Agent 3 (Code Analyzer)", anthropic_client)

    # Timestep 0
    initial_prompt = agent0.process(problem_description)
    initial_code = agent2.process(initial_prompt)
    initial_analysis = agent3.process(initial_code)
    initial_score = initial_analysis['score']
    initial_complexity = initial_analysis['time_complexity']

    current_prompt = initial_prompt
    current_code = initial_code
    current_score = initial_score
    current_complexity = initial_complexity

    # Subsequent timesteps
    no_improvement_count = 0
    for t in range(1, max_iterations + 1):
        print(f"\nIteration {t}")

        # Prompt optimization
        optimized_prompt = agent1.process(current_prompt, current_code, initial_analysis['feedback'])

        # Code generation with optimized prompt
        generated_code = agent2.process(optimized_prompt)

        # Code analysis
        analysis_result = agent3.process(generated_code)
        new_score = int(analysis_result['score'])
        print(f"Current score: {new_score}")

        if new_score <= current_score:
            no_improvement_count += 1
        else:
            no_improvement_count = 0
            current_prompt = optimized_prompt
            current_code = generated_code
            current_score = new_score
            current_complexity = analysis_result['time_complexity']

        if no_improvement_count >= improvement_threshold:
            print(f"No improvement for {improvement_threshold} consecutive steps. Stopping.")
            break

    print("\nWorkflow completed.")
    return {
        "problem_description": problem_description,
        "initial_prompt": initial_prompt,
        "optimized_prompt": current_prompt,
        "initial_code": initial_code,
        "final_code": current_code,
        "initial_score": initial_score,
        "final_score": current_score,
        "initial_time_complexity": initial_complexity,
        "final_time_complexity": current_complexity
    }

def create_prompt_optimization_dataset(problems: List[str], anthropic_client: Anthropic, n_samples: int) -> List[Dict[str, str]]:
    dataset = []
    selected_problems = random.sample(problems, min(n_samples, len(problems)))

    for i, problem in enumerate(selected_problems, 1):
        print(f"\nProcessing problem {i}/{len(selected_problems)}")
        result = agentic_workflow(problem, anthropic_client)
        dataset.append(result)

    return dataset

def save_dataset(dataset: List[Dict[str, str]], filename: str):
    with open(filename, 'w') as f:
        json.dump(dataset, f, indent=2)

# Main execution
def run_workflow(anthropic_api_key: str, leetcode_problems: List[str], n_samples: int = 25):
    anthropic_client = create_anthropic_client(anthropic_api_key)
    dataset = create_prompt_optimization_dataset(leetcode_problems, anthropic_client, n_samples)
    save_dataset(dataset, "prompt_optimization_dataset.json")
    print(f"\nDataset with {len(dataset)} samples has been generated and saved to 'prompt_optimization_dataset.json'")
    return dataset

# Example usage
anthropic_api_key = "Enter your secret key here"
leetcode_problems = list(task_list[:25])  # Your list of LeetCode problem descriptions
dataset = run_workflow(anthropic_api_key, leetcode_problems, n_samples=25)